# Stability Criteria: Output Comparison

Compares the **results** of the Roch natural and WEAC skier stability criteria on ECTP slabs:
success rates per pathway, output value distributions, and agreement on stability classification.

**Prerequisite:** run `stability_criteria_inputs.ipynb` first to generate the parquet files
this notebook loads (`roch_slab_results.parquet`, `weac_slab_results.parquet`, `total_slabs.txt`).

## Table of Contents

1. [Setup & Load Results](#1-setup--load-results)
2. [Success Rates](#2-success-rates)
3. [Output Value Distributions](#3-output-value-distributions)
4. [Classification Agreement](#4-classification-agreement)

| Criterion | Output | Threshold |
|-----------|--------|-----------|
| Roch natural | S_r = τ_c / τ | S_r < 1 → unstable |
| WEAC skier | g_delta = Γ/Γ_c | g_delta ≥ 1 → unstable |

> **Skier variant:** to use S_a = τ_c / (τ + δτ) with δτ = 150 N/m² (Föhn 1987),
> re-run `stability_criteria_inputs.ipynb` with the S_a block enabled and update
> column references below from `s_r`/`s_r_ok` to `s_a`/`s_a_ok`.

Nominal values only (output uncertainty not reported).

## 1. Setup & Load Results

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from notebook_utils import DENSITY_COLORS, rgba


In [2]:
RESULTS = Path('.')   # notebooks run from examples/

for fname in ('roch_slab_results.parquet', 'weac_slab_results.parquet', 'total_slabs.txt'):
    if not (RESULTS / fname).exists():
        raise FileNotFoundError(
            f'{fname} not found. Run stability_criteria_inputs.ipynb first.'
        )

roch_slab_df = pd.read_parquet(RESULTS / 'roch_slab_results.parquet', engine='pyarrow')
weac_slab_df = pd.read_parquet(RESULTS / 'weac_slab_results.parquet', engine='pyarrow')
total_slabs  = int((RESULTS / 'total_slabs.txt').read_text())

# ── Re-derive summary tables ──────────────────────────────────────────────
roch_cov = (
    roch_slab_df
    .groupby('density_method')
    .agg(
        n_density_ok=('slab_density_ok', 'sum'),
        n_s_r_ok    =('s_r_ok',          'sum'),
    )
    .reset_index()
)
roch_cov['pct_density'] = roch_cov['n_density_ok'] / total_slabs
roch_cov['pct_s_r']     = roch_cov['n_s_r_ok']     / total_slabs
roch_cov = roch_cov.sort_values('n_s_r_ok', ascending=False)

weac_cov = (
    weac_slab_df
    .groupby(['density_method', 'emod_method', 'nu_method'])
    .agg(
        n_density_ok =('ok_density',    'sum'),
        n_emod_ok    =('ok_emod',       'sum'),
        n_nu_ok      =('ok_nu',         'sum'),
        n_G_ok       =('ok_G',          'sum'),
        n_all_inputs =('all_inputs_ok', 'sum'),
        n_g_delta_ok =('g_delta_ok',    'sum'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)

print(f'Loaded {total_slabs:,} ECTP slabs')
print(f'Roch slab records:  {len(roch_slab_df):,}  ({len(roch_cov)} pathways)')
print(f'WEAC slab records:  {len(weac_slab_df):,}  ({len(weac_cov)} pathways)')

Loaded 14,776 ECTP slabs
Roch slab records:  59,104  (4 pathways)
WEAC slab records:  472,832  (32 pathways)


### Why is WEAC g_delta coverage near zero?

The success-rate table below will show that Roch S_r is valid for a large fraction of ECTP slabs,
while WEAC g_delta is valid for only 0–1% of slabs — even though ~5% of slabs have all required
WEAC inputs available (density, elastic modulus, Poisson's ratio, and a valid slope angle).

**This gap is expected and scientifically meaningful**, not a code defect:

- `stability_criteria_inputs.ipynb` applies a `weac_timeout_seconds=5.0` budget per slab.
  WEAC's eigensystem solver is iterative and can recurse deeply on numerically difficult inputs.
- Real-world ECTP slabs are often **thick and stiff** (high elastic modulus, large D11),
  making the coupled eigenvalue problem harder to converge than the synthetic geometries
  used in WEAC's original validation.
- Most failures are **solver timeouts or convergence failures**, not missing inputs.
- Roch S_r, by contrast, requires only density and is a simple closed-form formula — it has
  no convergence requirement.

The near-zero WEAC coverage is therefore a finding about the numerical difficulty of the
coupled criterion on field data, not about the coverage of the underlying inputs.

## 2. Success Rates

Fraction of ECTP slabs for which each criterion returns a valid stability output.

In [3]:
print('Roch Natural — Success Rate per Pathway')
print(f"  {'Method':<28}  {'N valid S_r':>30}")
print(f"  {'-'*62}")
for _, row in roch_cov.iterrows():
    n = int(row['n_s_r_ok'])
    print(f"  {row['density_method']:<28}  {n:>5} / {total_slabs} ({row['pct_s_r']:.1%})")

print()
print('WEAC Skier — Success Rate per Pathway (all 32)')
print(f"  {'Pathway':<55}  {'N valid g_delta':>30}")
print(f"  {'-'*90}")
for _, row in weac_cov.iterrows():
    pw = f"{row['density_method']} → {row['emod_method']} → {row['nu_method']}"
    n  = int(row['n_g_delta_ok'])
    print(f"  {pw:<55}  {n:>5} / {total_slabs} ({n / total_slabs:.1%})")

Roch Natural — Success Rate per Pathway
  Method                                           N valid S_r
  --------------------------------------------------------------
  kim_jamieson_table2            5475 / 14776 (37.1%)
  geldsetzer                     4200 / 14776 (28.4%)
  kim_jamieson_table5            1066 / 14776 (7.2%)
  data_flow                       100 / 14776 (0.7%)

WEAC Skier — Success Rate per Pathway (all 32)
  Pathway                                                                 N valid g_delta
  ------------------------------------------------------------------------------------------
  kim_jamieson_table2 → wautier → kochle                       6 / 14776 (0.0%)
  kim_jamieson_table2 → schottner → kochle                     9 / 14776 (0.1%)
  geldsetzer → wautier → kochle                                6 / 14776 (0.0%)
  geldsetzer → schottner → kochle                              9 / 14776 (0.1%)
  geldsetzer → kochle → kochle                                 8 / 

## 3. Output Value Distributions

Violin plots of S_r (Roch) and g_delta (WEAC) across pathways.
Threshold lines mark the instability boundary for each criterion.

In [4]:
roch_valid   = roch_slab_df[roch_slab_df['s_r_ok']].copy()
methods_by_n = (
    roch_valid.groupby('density_method')['slab_id']
    .count().sort_values(ascending=False).index.tolist()
)

fig = go.Figure()
for dm in reversed(methods_by_n):
    vals  = roch_valid.loc[roch_valid['density_method'] == dm, 's_r'].dropna().values
    if len(vals) == 0:
        continue
    n     = len(vals)
    color = DENSITY_COLORS.get(dm, '#888')
    fig.add_trace(go.Violin(
        x=vals,
        y=[f"{dm}  <i>({n:,}, {n / total_slabs:.1%})</i>"] * n,
        orientation='h',
        name=dm,
        box_visible=True, meanline_visible=True, points=False,
        fillcolor=rgba(color, 0.65),
        line_color=rgba(color, 1.0),
        showlegend=False,
    ))

fig.add_vline(
    x=1.0, line_dash='dash',
    line_color='rgba(196,84,78,0.9)', line_width=2,
    annotation_text='S_r = 1 (instability)', annotation_position='top right',
)
fig.update_layout(
    title=dict(
        text=(
            '<b>Roch Natural — S_r Distribution by Density Method</b><br>'
            '<sup>Nominal values — ordered top→bottom by coverage — '
            'box = IQR, line = mean</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    xaxis=dict(title='S_r  (< 1 → unstable)', gridcolor='rgba(200,200,200,0.4)'),
    yaxis=dict(autorange='reversed', tickfont=dict(size=10)),
    plot_bgcolor='white',
    violingap=0.10,
    width=820, height=340,
    margin=dict(l=290, r=40, t=90, b=60),
)
fig.show()

In [5]:
weac_valid = weac_slab_df[weac_slab_df['g_delta_ok']].copy()
weac_valid['pathway'] = (
    weac_valid['density_method'] + ' → ' +
    weac_valid['emod_method']    + ' → ' +
    weac_valid['nu_method']
)
pw_by_n = (
    weac_valid.groupby('pathway')['slab_id']
    .count().sort_values(ascending=False).index.tolist()
)

fig = go.Figure()
for pw_label in reversed(pw_by_n):
    vals = weac_valid.loc[weac_valid['pathway'] == pw_label, 'g_delta'].dropna().values
    if len(vals) == 0:
        continue
    n     = len(vals)
    dm    = pw_label.split(' → ')[0]
    color = DENSITY_COLORS.get(dm, '#888')
    fig.add_trace(go.Violin(
        x=vals,
        y=[f"{pw_label}  <i>({n:,}, {n / total_slabs:.1%})</i>"] * n,
        orientation='h',
        name=pw_label,
        box_visible=True, meanline_visible=True, points=False,
        fillcolor=rgba(color, 0.55),
        line_color=rgba(color, 0.90),
        showlegend=False,
    ))

fig.add_vline(
    x=1.0, line_dash='dash',
    line_color='rgba(196,84,78,0.9)', line_width=2,
    annotation_text='g_delta = 1 (instability)', annotation_position='top right',
)
fig.update_layout(
    title=dict(
        text=(
            '<b>WEAC Skier — g_delta Distribution by Pathway</b><br>'
            '<sup>Nominal values — ordered top→bottom by coverage — '
            'box = IQR, line = mean — colour = density method</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    xaxis=dict(title='g_delta  (Γ/Γ_c, ≥ 1 → unstable)', gridcolor='rgba(200,200,200,0.4)'),
    yaxis=dict(autorange='reversed', tickfont=dict(size=9.5)),
    plot_bgcolor='white',
    violingap=0.06,
    width=820, height=720,
    margin=dict(l=420, r=40, t=90, b=60),
)
fig.show()

## 4. Classification Agreement

For slabs where **both** criteria return a valid result (matched on `slab_id` + `density_method`),
compare whether they agree on stability classification.

Multiple WEAC pathways per slab are included (one row per slab × density method × E-mod method × ν method).

In [6]:
roch_ok = roch_slab_df[roch_slab_df['s_r_ok']][
    ['slab_id', 'density_method', 's_r', 'unstable_roch']
].copy()
weac_ok = weac_slab_df[weac_slab_df['g_delta_ok']][
    ['slab_id', 'density_method', 'emod_method', 'nu_method', 'g_delta', 'unstable_weac']
].copy()

matched = roch_ok.merge(weac_ok, on=['slab_id', 'density_method'])
print(f'Slabs with valid Roch AND WEAC: {len(matched):,}')
print('  (matched on slab_id + density_method; multiple WEAC pathways per slab included)')

if len(matched) == 0:
    print('\nNo matched slabs — no comparison possible.')
else:
    matched['roch_class'] = matched['unstable_roch'].map(
        {True: 'Unstable (S_r<1)', False: 'Stable (S_r≥1)'}
    )
    matched['weac_class'] = matched['unstable_weac'].map(
        {True: 'Unstable (g_δ≥1)', False: 'Stable (g_δ<1)'}
    )

    confusion = pd.crosstab(
        matched['roch_class'], matched['weac_class'],
        rownames=['Roch S_r'], colnames=['WEAC g_delta'],
        margins=True,
    )
    print()
    print('Classification agreement (Roch rows × WEAC columns):')
    print(confusion.to_string())

    n_agree   = (matched['unstable_roch'] == matched['unstable_weac']).sum()
    pct_agree = n_agree / len(matched)
    print(f'\nAgreement: {n_agree} / {len(matched)} ({pct_agree:.1%})')

Slabs with valid Roch AND WEAC: 131
  (matched on slab_id + density_method; multiple WEAC pathways per slab included)

Classification agreement (Roch rows × WEAC columns):
WEAC g_delta    Stable (g_δ<1)  Unstable (g_δ≥1)  All
Roch S_r                                             
Stable (S_r≥1)              47                84  131
All                         47                84  131

Agreement: 47 / 131 (35.9%)


In [7]:
if len(matched) == 0:
    print('No matched slabs — scatter plot skipped.')
else:
    agree    = matched[matched['unstable_roch'] == matched['unstable_weac']]
    disagree = matched[matched['unstable_roch'] != matched['unstable_weac']]

    fig = go.Figure()

    for sub, color, label in [
        (agree,    '#009E73', 'Agreement'),
        (disagree, '#D55E00', 'Disagreement'),
    ]:
        if len(sub) == 0:
            continue
        hover = [
            f"slab: {r['slab_id']}<br>"
            f"density: {r['density_method']}<br>"
            f"E-mod: {r['emod_method']}<br>"
            f"S_r={r['s_r']:.3f}  g_delta={r['g_delta']:.3f}"
            for _, r in sub.iterrows()
        ]
        fig.add_trace(go.Scatter(
            x=sub['s_r'].values,
            y=sub['g_delta'].values,
            mode='markers',
            marker=dict(
                color=rgba(color, 0.70), size=7,
                line=dict(width=0.8, color=rgba(color, 1.0)),
            ),
            name=f'{label} (n={len(sub)})',
            text=hover, hoverinfo='text',
        ))

    fig.add_vline(
        x=1.0, line_dash='dash',
        line_color='rgba(100,100,100,0.5)', line_width=1.5,
        annotation_text='S_r = 1', annotation_position='top right',
    )
    fig.add_hline(
        y=1.0, line_dash='dash',
        line_color='rgba(100,100,100,0.5)', line_width=1.5,
        annotation_text='g_delta = 1', annotation_position='top right',
    )

    fig.update_layout(
        title=dict(
            text=(
                '<b>Roch S_r vs WEAC g_delta — Matched Slabs</b><br>'
                '<sup>Matched by slab_id + density_method — '
                'green = criteria agree, orange = disagree</sup>'
            ),
            x=0.5, xanchor='center', font=dict(size=13),
        ),
        xaxis=dict(
            title='Roch S_r  (< 1 → unstable)',
            gridcolor='rgba(200,200,200,0.4)',
        ),
        yaxis=dict(
            title='WEAC g_delta  (≥ 1 → unstable)',
            gridcolor='rgba(200,200,200,0.4)',
        ),
        plot_bgcolor='white',
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)'),
        width=700, height=600,
        margin=dict(l=70, r=40, t=90, b=60),
    )
    fig.show()